In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
df.shape

In [ ]:
df = pd.read_csv(path + "/Q1_data.csv")
df.head()


In [ ]:
df.info()

In [ ]:
df.describe()


In [ ]:
df.columns


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.hist(df["Delivery_Time"], bins=30)
plt.xlabel("Delivery Time (minutes)")
plt.ylabel("Frequency")
plt.title("Distribution of Delivery Time")
plt.show()


In [ ]:
if "Order_ID" in df.columns:
    df = df.drop(columns=["Order_ID"])


In [ ]:
df.isna().sum()


In [ ]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
cat_cols = df.select_dtypes(include=["object"]).columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])


In [ ]:
df = df.drop_duplicates()


In [ ]:
df.duplicated().sum()


In [ ]:
df = pd.get_dummies(df, drop_first=True)


In [ ]:
df.drop(columns=["Delivery_Time"])


In [ ]:
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=["Delivery_Time"])
y = df["Delivery_Time"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X = pd.DataFrame(X_scaled, columns=X.columns)


In [ ]:
possible_targets = ["delivery_time", "delivery time", "Delivery_Time", "Delivery Time"]

target_col = None
for c in possible_targets:
    if c in df.columns:
        target_col = c
        break

if target_col is None:
    raise ValueError(f"Target column not found. Available columns: {list(df.columns)}")

X = df.drop(columns=[target_col])
y = df[target_col]

print("Target column:", target_col)
print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_val)

    mae = mean_absolute_error(y_val, preds)
    mae_scores.append(mae)

    print(f"Fold {fold} MAE: {mae:.4f}")

print("\nAverage MAE across all folds:", np.mean(mae_scores))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor

final_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)
final_model.fit(X, y)

importances = final_model.feature_importances_
feature_names = np.array(X.columns)

idx = np.argsort(importances)[-15:]

plt.figure(figsize=(10,6))
plt.barh(feature_names[idx], importances[idx])
plt.xlabel("Importance")
plt.title("Top 15 Feature Importances (Random Forest)")
plt.show()


In [ ]:
y_pred = final_model.predict(X)

plt.figure(figsize=(8,5))
plt.hist(y_pred, bins=30)
plt.xlabel("Predicted Delivery Time")
plt.ylabel("Frequency")
plt.title("Predicted Delivery Time Distribution")
plt.show()


In [ ]:
%pip install catboost

In [ ]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor

kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    rf = RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )
    cat = CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        random_state=42,
        verbose=0
    )

    rf.fit(X_train, y_train)
    cat.fit(X_train, y_train)

    rf_pred = rf.predict(X_val)
    cat_pred = cat.predict(X_val)

    ensemble_pred = (rf_pred + cat_pred) / 2

    mae = mean_absolute_error(y_val, ensemble_pred)
    mae_scores.append(mae)

    print(f"Fold {fold} Ensemble MAE: {mae:.4f}")

print("\nAverage Ensemble MAE across all folds:", np.mean(mae_scores))
